# 03 - Feature Comprehension

## Obiettivo del notebook

In questo notebook analizziamo il significato delle feature disponibili nel dataset e il loro possibile ruolo nella previsione di `damage_grade`.

L'obiettivo non è ancora costruire un modello, ma comprendere:

- quali gruppi di variabili sono presenti;
- quale significato hanno le feature;
- quali feature sembrano più informative rispetto al danno;
- quali indicazioni possiamo ricavare per preprocessing e modellazione.

Questo notebook serve quindi da ponte tra l'analisi esplorativa / qualità dei dati e la futura fase di preprocessing e baseline modeling.

È importante precisare che le relazioni osservate in questo notebook sono di tipo esplorativo: non misurano ancora l'impatto effettivo delle feature sulle performance di un modello, ma aiutano a formulare ipotesi motivate per preprocessing e modellazione.

In [1]:
import pandas as pd

## 1. Caricamento dei dati

Carichiamo i dati di training e uniamo le feature con la variabile target `damage_grade` usando `building_id`.

In [2]:
train_values = pd.read_csv("../data/raw/train_values.csv")
train_labels = pd.read_csv("../data/raw/train_labels.csv")

train = train_values.merge(train_labels, on="building_id")

train.head()

,building_id,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,foundation_type,...,has_secondary_use_hotel,has_secondary_use_rental,has_secondary_use_institution,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other,damage_grade
0,802906,6,487,12198,2,30,6,5,t,r,...,0,0,0,0,0,0,0,0,0,3
1,28830,8,900,2812,2,10,8,7,o,r,...,0,0,0,0,0,0,0,0,0,2
2,94947,21,363,8973,2,10,5,5,t,r,...,0,0,0,0,0,0,0,0,0,3
3,590882,22,418,10694,2,10,6,5,t,r,...,0,0,0,0,0,0,0,0,0,2
4,201944,11,131,1488,3,30,8,9,t,r,...,0,0,0,0,0,0,0,0,0,3


## 2. Definizione dei gruppi di feature

Le feature vengono suddivise in gruppi coerenti dal punto di vista semantico.

Questa distinzione è importante perché gruppi diversi richiedono trattamenti diversi nella pipeline di preprocessing.

In [3]:
geo_features = [
    "geo_level_1_id",
    "geo_level_2_id",
    "geo_level_3_id"
]

numeric_features = [
    "count_floors_pre_eq",
    "age",
    "area_percentage",
    "height_percentage",
    "count_families"
]

categorical_features = [
    "land_surface_condition",
    "foundation_type",
    "roof_type",
    "ground_floor_type",
    "other_floor_type",
    "position",
    "plan_configuration",
    "legal_ownership_status"
]

superstructure_features = [
    col for col in train.columns
    if col.startswith("has_superstructure_")
]

secondary_use_features = [
    col for col in train.columns
    if col.startswith("has_secondary_use")
]

feature_groups_summary = pd.DataFrame({
    "group": [
        "geographical",
        "structural_numeric",
        "structural_categorical",
        "superstructure_binary",
        "secondary_use_binary"
    ],
    "n_features": [
        len(geo_features),
        len(numeric_features),
        len(categorical_features),
        len(superstructure_features),
        len(secondary_use_features)
    ]
})

feature_groups_summary

,group,n_features
0,geographical,3
1,structural_numeric,5
2,structural_categorical,8
3,superstructure_binary,11
4,secondary_use_binary,11


### Interpretazione dei gruppi di feature

La suddivisione evidenzia che il dataset contiene gruppi di feature semanticamente molto diversi.

Le variabili geografiche sono poche, ma possono contenere informazione territoriale rilevante. Le variabili numeriche descrivono dimensione, età e struttura generale dell'edificio. Le variabili categoriche rappresentano caratteristiche costruttive e configurazioni dell'edificio.

È importante separare le feature binarie di superstruttura dalle feature binarie di uso secondario. Le prime descrivono direttamente materiali e tecniche costruttive, quindi sono legate alla vulnerabilità fisica dell'edificio. Le seconde descrivono invece l'utilizzo dell'edificio e potrebbero avere un ruolo più debole o indiretto.

Questa distinzione guiderà le scelte successive di preprocessing e modellazione.


## 3. Feature map

Costruiamo una tabella riassuntiva che associa ogni variabile a:

- gruppo di appartenenza;
- tipo di variabile;
- descrizione semantica;
- possibile ruolo nel modello.

Questa tabella sarà utile anche per aggiornare `docs/feature_notes.md`.

In [4]:
feature_map = []

for col in train.columns:
    if col == "building_id":
        group = "identifier"
        var_type = "id"
        description = "Identificativo univoco dell'edificio."
        modeling_role = "Da escludere dal modello."
    elif col in geo_features:
        group = "geographical"
        var_type = "categorical_id"
        description = "Identificatore geografico a diverso livello di granularità."
        modeling_role = "Potenzialmente molto informativo, da trattare come categorico."
    elif col in numeric_features:
        group = "structural_numeric"
        var_type = "numeric"
        description = "Caratteristica numerica legata alla struttura o dimensione dell'edificio."
        modeling_role = "Utile come segnale strutturale, probabilmente non sufficiente da sola."
    elif col in categorical_features:
        group = "structural_categorical"
        var_type = "categorical"
        description = "Caratteristica categorica legata alla struttura o alla configurazione dell'edificio."
        modeling_role = "Potenzialmente molto informativa, richiede encoding."
    elif col in superstructure_features:
        group = "superstructure_binary"
        var_type = "binary"
        description = "Presenza o assenza di un materiale / tipo di superstruttura."
        modeling_role = "Molto rilevante per stimare la vulnerabilità dell'edificio."
    elif col in secondary_use_features:
        group = "secondary_use_binary"
        var_type = "binary"
        description = "Presenza o assenza di uno specifico uso secondario dell'edificio."
        modeling_role = "Possibile segnale ausiliario, da valutare empiricamente."
    elif col == "damage_grade":
        group = "target"
        var_type = "target"
        description = "Livello di danno da predire."
        modeling_role = "Variabile target."
    else:
        group = "other"
        var_type = "unknown"
        description = "Variabile da verificare."
        modeling_role = "Da valutare."

    feature_map.append({
        "feature": col,
        "group": group,
        "type": var_type,
        "description": description,
        "modeling_role": modeling_role
    })

feature_map_df = pd.DataFrame(feature_map)
feature_map_df

,feature,group,type,description,modeling_role
0,building_id,identifier,id,Identificativo univoco dell'edificio.,Da escludere dal modello.
1,geo_level_1_id,geographical,categorical_id,Identificatore geografico a diverso livello di...,"Potenzialmente molto informativo, da trattare ..."
2,geo_level_2_id,geographical,categorical_id,Identificatore geografico a diverso livello di...,"Potenzialmente molto informativo, da trattare ..."
3,geo_level_3_id,geographical,categorical_id,Identificatore geografico a diverso livello di...,"Potenzialmente molto informativo, da trattare ..."
4,count_floors_pre_eq,structural_numeric,numeric,Caratteristica numerica legata alla struttura ...,"Utile come segnale strutturale, probabilmente ..."
5,age,structural_numeric,numeric,Caratteristica numerica legata alla struttura ...,"Utile come segnale strutturale, probabilmente ..."
6,area_percentage,structural_numeric,numeric,Caratteristica numerica legata alla struttura ...,"Utile come segnale strutturale, probabilmente ..."
7,height_percentage,structural_numeric,numeric,Caratteristica numerica legata alla struttura ...,"Utile come segnale strutturale, probabilmente ..."
8,land_surface_condition,structural_categorical,categorical,Caratteristica categorica legata alla struttur...,"Potenzialmente molto informativa, richiede enc..."
9,foundation_type,structural_categorical,categorical,Caratteristica categorica legata alla struttur...,"Potenzialmente molto informativa, richiede enc..."


### Nota metodologica sulla feature map

La feature map permette di rendere esplicite le assunzioni sulle variabili prima della modellazione.

Un aspetto importante è che non tutte le colonne numeriche devono essere interpretate come variabili continue. Ad esempio, `geo_level_1_id`, `geo_level_2_id` e `geo_level_3_id` sono codici identificativi geografici: anche se sono rappresentati da numeri, il loro valore non ha un significato quantitativo ordinato.

Questa distinzione è fondamentale per evitare trasformazioni scorrette nella pipeline di preprocessing.


## 4. Analisi delle feature numeriche

Analizziamo le feature numeriche strutturali rispetto a `damage_grade`.

L'obiettivo è capire se variabili come età, altezza, area e numero di piani mostrano differenze tra i livelli di danno.

In [5]:
train.groupby("damage_grade")[numeric_features].mean()

,count_floors_pre_eq,age,area_percentage,height_percentage,count_families
damage_grade,,,,,
1,1.841307,17.320490,9.716009,5.147349,0.915101
2,2.131763,27.342118,8.034049,5.432345,0.982018
3,2.209338,27.817423,7.501743,5.520477,1.007063


### Interpretazione dei risultati

Le feature numeriche mostrano alcune differenze tra i livelli di danno, ma il segnale appare moderato.

L'età media degli edifici aumenta passando dal danno basso al danno elevato: gli edifici con `damage_grade = 1` hanno età media circa 17.3, mentre quelli con `damage_grade = 3` arrivano a circa 27.8. Questo suggerisce che gli edifici più vecchi possano essere mediamente più vulnerabili.

Anche `count_floors_pre_eq` e `height_percentage` aumentano leggermente con il livello di danno, indicando una possibile relazione tra dimensione verticale dell'edificio e vulnerabilità. Tuttavia, le differenze non sono estremamente marcate.

`area_percentage` mostra invece un andamento inverso: gli edifici meno danneggiati hanno area media maggiore, mentre quelli più danneggiati hanno area media più bassa. Questo pattern potrebbe dipendere da differenze strutturali o territoriali e andrà verificato nelle fasi successive.

`count_families` mostra variazioni molto contenute e sembra meno discriminante rispetto alle altre feature numeriche.

In sintesi, le feature numeriche forniscono un segnale utile ma non sufficiente da sole per distinguere chiaramente i livelli di danno.


## 5. Analisi delle feature geografiche

Le feature geografiche sono codici identificativi, non variabili numeriche continue.

Per questo motivo non devono essere interpretate come quantità ordinate. Il loro possibile valore predittivo deriva dal fatto che aree diverse possono avere caratteristiche costruttive, esposizione e vulnerabilità diverse.

In [6]:
geo_cardinality = train[geo_features].nunique().to_frame(name="n_unique_values")
geo_cardinality

,n_unique_values
geo_level_1_id,31
geo_level_2_id,1414
geo_level_3_id,11595


In [7]:
geo_level_1_damage_distribution = pd.crosstab(
    train["geo_level_1_id"],
    train["damage_grade"],
    normalize="index"
)

geo_level_1_damage_distribution.head()

damage_grade,1,2,3
geo_level_1_id,,,
0,0.084019,0.766642,0.149339
1,0.152166,0.734913,0.112921
2,0.091300,0.655209,0.253491
3,0.032493,0.603448,0.364058
4,0.035763,0.766337,0.197900


### Interpretazione dei risultati

Le feature geografiche hanno una natura particolare: sono codici identificativi e non quantità numeriche continue. Per questo motivo non devono essere interpretate come valori ordinati.

La cardinalità cresce molto passando da `geo_level_1_id` a `geo_level_3_id`: il primo livello contiene 31 valori distinti, mentre il terzo supera gli 11.000 valori. Questo indica che le variabili geografiche possono contenere un segnale informativo rilevante, ma richiedono attenzione nella codifica.

La distribuzione del target varia tra aree geografiche diverse. Già osservando i primi valori di `geo_level_1_id`, la quota di edifici con danno elevato cambia sensibilmente. Questo suggerisce che la posizione geografica possa catturare differenze territoriali, caratteristiche costruttive locali o diversa intensità dell'evento sismico.

Dal punto di vista del preprocessing, `geo_level_1_id` può essere gestita più facilmente con encoding categorico standard. Per `geo_level_2_id` e `geo_level_3_id`, invece, la cardinalità elevata rende necessario valutare strategie più attente, come frequency encoding o target encoding.

Eventuali tecniche di target encoding dovranno però essere applicate esclusivamente all'interno della procedura di validazione, evitando di usare informazioni del validation set durante il fitting dell'encoding. In caso contrario si introdurrebbe data leakage.


## 6. Analisi delle feature categoriche strutturali

Le feature categoriche descrivono aspetti costruttivi e configurazioni dell'edificio.

Per ciascuna variabile analizziamo la distribuzione normalizzata di `damage_grade`, così da capire se alcune categorie sono associate più spesso a danni bassi, medi o elevati.

In [8]:
for col in categorical_features:
    print(f"\n=== {col} ===")
    display(pd.crosstab(train[col], train["damage_grade"], normalize="index"))


=== land_surface_condition ===


damage_grade,1,2,3
land_surface_condition,,,
n,0.071943,0.604706,0.323351
o,0.072391,0.566138,0.361472
t,0.101339,0.563151,0.335509



=== foundation_type ===


damage_grade,1,2,3
foundation_type,,,
h,0.247238,0.399862,0.352901
i,0.567539,0.411570,0.020890
r,0.048906,0.572615,0.378479
u,0.258696,0.598948,0.142356
w,0.287935,0.613176,0.098889



=== roof_type ===


...

damage_grade,1,2,3
position,,,
j,0.126111,0.594715,0.279175
o,0.051436,0.689670,0.258894
s,0.098308,0.574254,0.327438
t,0.080707,0.529187,0.390106



=== plan_configuration ===


damage_grade,1,2,3
plan_configuration,,,
a,0.261905,0.623016,0.115079
c,0.264615,0.633846,0.101538
d,0.093213,0.569928,0.336859
f,0.000000,0.727273,0.272727
m,0.173913,0.739130,0.086957
n,0.157895,0.526316,0.315789
o,0.251572,0.603774,0.144654
q,0.137210,0.454146,0.408644
s,0.153179,0.644509,0.202312



=== legal_ownership_status ===


damage_grade,1,2,3
legal_ownership_status,,,
a,0.274129,0.557692,0.168179
r,0.144603,0.493551,0.361847
v,0.092732,0.570473,0.336795
w,0.048562,0.487112,0.464326


### Conteggi assoluti delle feature categoriche

Le tabelle precedenti sono normalizzate per riga e mostrano quindi percentuali. Questo è utile per confrontare la distribuzione di `damage_grade` tra categorie diverse, ma può essere fuorviante se alcune categorie sono poco rappresentate.

Per questo motivo affianchiamo alle percentuali anche i conteggi assoluti delle categorie. In questo modo possiamo distinguere pattern robusti da possibili effetti dovuti a categorie rare.


In [9]:
for col in categorical_features:
    print(f"\n=== {col} ===")
    
    category_counts = train[col].value_counts(dropna=False).rename("count")
    category_percentages = train[col].value_counts(normalize=True, dropna=False).rename("percentage")
    
    category_summary = pd.concat(
        [category_counts, category_percentages],
        axis=1
    ).sort_index()
    
    display(category_summary)



=== land_surface_condition ===


,count,percentage
land_surface_condition,,
n,35528,0.136331
o,8316,0.031911
t,216757,0.831758



=== foundation_type ===


,count,percentage
foundation_type,,
h,1448,0.005556
i,10579,0.040595
r,219196,0.841117
u,14260,0.054720
w,15118,0.058012



=== roof_type ===


...

,count,percentage
position,,
j,13282,0.050967
o,2333,0.008952
s,202090,0.775477
t,42896,0.164604



=== plan_configuration ===


,count,percentage
plan_configuration,,
a,252,0.000967
c,325,0.001247
d,250072,0.959597
f,22,0.000084
m,46,0.000177
n,38,0.000146
o,159,0.000610
q,5692,0.021842
s,346,0.001328



=== legal_ownership_status ===


,count,percentage
legal_ownership_status,,
a,5512,0.021151
r,1473,0.005652
v,250939,0.962924
w,2677,0.010272


### Interpretazione dei risultati

Le feature categoriche strutturali mostrano livelli di informatività diversi.

`land_surface_condition` presenta differenze abbastanza contenute tra le categorie. La quota di danno elevato varia, ma non in modo tale da separare chiaramente le classi. Questa feature potrebbe contribuire al modello, ma non sembra tra le più discriminanti.

`foundation_type` è una delle feature categoriche più informative. Alcune categorie mostrano una distribuzione molto diversa del danno: la categoria `i` è fortemente associata a danni bassi, mentre la categoria `r` presenta una quota molto più alta di danni elevati. Questo suggerisce che il tipo di fondazione sia un elemento importante nella vulnerabilità strutturale dell'edificio.

`roof_type` mostra una chiara relazione con il livello di danno. La categoria `x` è associata a una percentuale molto bassa di danni elevati, mentre `n` e `q` presentano una quota decisamente maggiore di edifici gravemente danneggiati.

`ground_floor_type` appare molto informativa. La categoria `v` è associata soprattutto a danni bassi, mentre categorie come `f` e `x` mostrano una maggiore incidenza di danni elevati.

`other_floor_type` mostra differenze rilevanti tra categorie. La categoria `s` è associata a edifici meno danneggiati, mentre `q` e `x` mostrano quote più alte di danno elevato.

`position` mostra alcune differenze tra categorie, ma il segnale sembra meno netto rispetto a feature come `foundation_type`, `roof_type` o `ground_floor_type`. Può comunque contribuire al modello, ma probabilmente non sarà tra le feature più determinanti.

`plan_configuration` mostra alcuni pattern interessanti, ma l'interpretazione deve essere cauta. Alcune categorie hanno percentuali apparentemente molto diverse, ma potrebbero essere poco rappresentate nel dataset. Per questa feature sarà utile affiancare alle distribuzioni normalizzate anche i conteggi assoluti delle categorie.

`legal_ownership_status` mostra differenze non trascurabili tra categorie. Tuttavia, il suo significato rispetto al danno strutturale è meno diretto: potrebbe catturare informazioni indirette legate al tipo di edificio, al contesto territoriale o ad altri fattori socio-economici.

In sintesi, le variabili più rilevanti sembrano essere `foundation_type`, `roof_type`, `ground_floor_type` e `other_floor_type`, perché mostrano differenze marcate nella distribuzione di `damage_grade`. Questa sezione conferma che le caratteristiche costruttive dell'edificio sono uno dei gruppi di feature più importanti per la previsione del danno.


## 7. Analisi delle feature binarie di superstruttura

Le variabili `has_superstructure_*` indicano la presenza o assenza di materiali o tecniche costruttive.

Sono particolarmente importanti perché descrivono direttamente la struttura fisica dell'edificio.

In [10]:
train.groupby("damage_grade")[superstructure_features].mean().T.sort_values(by=3, ascending=False)

damage_grade,1,2,3
has_superstructure_mud_mortar_stone,0.348671,0.768567,0.869706
has_superstructure_timber,0.304171,0.271059,0.213500
has_superstructure_adobe_mud,0.023643,0.093768,0.098661
has_superstructure_mud_mortar_brick,0.024797,0.078902,0.062372
has_superstructure_bamboo,0.113000,0.094436,0.060928
has_superstructure_stone_flag,0.007244,0.030582,0.048511
has_superstructure_rc_non_engineered,0.153519,0.039337,0.016166
has_superstructure_cement_mortar_brick,0.282797,0.077034,0.012486
has_superstructure_other,0.026110,0.015163,0.011477
has_superstructure_cement_mortar_stone,0.032996,0.021908,0.007739


### Interpretazione dei risultati

Le feature binarie di superstruttura risultano tra le più informative del dataset.

La presenza di `has_superstructure_mud_mortar_stone` aumenta drasticamente al crescere del livello di danno: è presente in circa il 34.9% degli edifici con danno basso, ma in circa l'87.0% degli edifici con danno elevato. Questo suggerisce una forte associazione tra questa tipologia strutturale e vulnerabilità sismica.

Al contrario, alcune tecniche o materiali più resistenti risultano molto più frequenti negli edifici con danno basso. `has_superstructure_rc_engineered` è presente in circa il 10.6% degli edifici con danno basso, ma quasi assente negli edifici con danno elevato. Anche `has_superstructure_cement_mortar_brick` mostra un andamento simile, passando da circa 28.3% nei danni bassi a circa 1.2% nei danni elevati.

Feature come `timber` e `bamboo` diminuiscono al crescere del danno, suggerendo una possibile associazione con livelli di danno meno severi. Tuttavia, queste variabili possono coesistere con altri materiali, quindi l'interpretazione va completata nella fase di modellazione multivariata.

Nel complesso, le feature di superstruttura devono essere mantenute nella pipeline perché rappresentano direttamente materiali e tecniche costruttive dell'edificio.


## 8. Analisi delle feature binarie di uso secondario

Le variabili `has_secondary_use_*` indicano eventuali usi secondari dell'edificio.

Queste feature potrebbero essere meno direttamente legate alla vulnerabilità strutturale, ma possono comunque catturare differenze nel tipo di edificio.

In [11]:
train.groupby("damage_grade")[secondary_use_features].mean().T.sort_values(by=3, ascending=False)

damage_grade,1,2,3
has_secondary_use,0.169081,0.119487,0.082472
has_secondary_use_agriculture,0.032996,0.072029,0.060412
has_secondary_use_hotel,0.088203,0.034116,0.017072
has_secondary_use_other,0.006965,0.005747,0.003520
has_secondary_use_rental,0.034907,0.007089,0.002098
has_secondary_use_industry,0.002348,0.001025,0.000780
has_secondary_use_institution,0.003940,0.000870,0.000195
has_secondary_use_school,0.001194,0.000317,0.000195
has_secondary_use_use_police,0.000159,0.000081,0.000080
has_secondary_use_health_post,0.000478,0.000216,0.000057


### Interpretazione dei risultati

Le feature di uso secondario sembrano avere un segnale più debole e indiretto rispetto alle feature strutturali.

La variabile generale `has_secondary_use` è più frequente negli edifici con danno basso e diminuisce nei livelli di danno più elevati. Un andamento simile si osserva per usi come `hotel` e `rental`, che risultano più presenti negli edifici meno danneggiati.

Questo pattern non deve essere interpretato necessariamente come un effetto causale dell'uso secondario sul danno. È più plausibile che queste variabili riflettano differenze nel tipo di edificio, nel contesto urbano o rurale, nella qualità costruttiva o nella localizzazione geografica.

Molte feature di uso secondario hanno frequenze molto basse, come `school`, `health_post`, `gov_office` o `use_police`. Queste variabili potrebbero avere scarso impatto predittivo individuale, ma conviene mantenerle inizialmente e lasciare che la fase di modellazione ne valuti l'utilità.

Nel complesso, le feature di uso secondario vanno considerate come segnali ausiliari, non come fattori principali di vulnerabilità.


## 9. Sintesi dell'informatività attesa

Costruiamo una sintesi qualitativa dei gruppi di feature, utile per collegare questa analisi alla fase successiva di preprocessing e modellazione.

In [12]:
feature_relevance_summary = pd.DataFrame({
    "feature_group": [
        "identifier",
        "geographical",
        "structural_numeric",
        "structural_categorical",
        "superstructure_binary",
        "secondary_use_binary"
    ],
    "expected_relevance": [
        "none",
        "high",
        "medium",
        "high",
        "high",
        "low-medium"
    ],
    "suggested_preprocessing": [
        "drop",
        "categorical encoding / possible target encoding later",
        "keep numeric; scaling optional depending on model",
        "categorical encoding",
        "keep as binary",
        "keep as binary; evaluate usefulness"
    ],
    "notes": [
        "Identificativo tecnico, non informativo.",
        "Può catturare vulnerabilità territoriali e differenze locali.",
        "Fornisce informazioni su dimensione ed età, ma con possibile sovrapposizione tra classi.",
        "Descrive aspetti costruttivi rilevanti.",
        "Descrive direttamente materiali e struttura.",
        "Segnale ausiliario, probabilmente meno centrale."
    ]
})

feature_relevance_summary

,feature_group,expected_relevance,suggested_preprocessing,notes
0,identifier,none,drop,"Identificativo tecnico, non informativo."
1,geographical,high,categorical encoding / possible target encodin...,Può catturare vulnerabilità territoriali e dif...
2,structural_numeric,medium,keep numeric; scaling optional depending on model,"Fornisce informazioni su dimensione ed età, ma..."
3,structural_categorical,high,categorical encoding,Descrive aspetti costruttivi rilevanti.
4,superstructure_binary,high,keep as binary,Descrive direttamente materiali e struttura.
5,secondary_use_binary,low-medium,keep as binary; evaluate usefulness,"Segnale ausiliario, probabilmente meno centrale."


### Interpretazione della sintesi

La sintesi conferma che i gruppi più promettenti sono le feature geografiche, le feature categoriche strutturali e le feature binarie di superstruttura.

Le feature numeriche mostrano un segnale interpretabile, ma meno marcato. Età, numero di piani, altezza e area possono contribuire alla previsione, ma difficilmente saranno sufficienti da sole per separare chiaramente i livelli di danno.

Le feature di uso secondario sembrano invece più deboli e indirette. Verranno mantenute nella prima pipeline, ma il loro contributo dovrà essere verificato empiricamente durante la modellazione.

Questa classificazione non sostituisce la valutazione tramite modello, ma fornisce una guida motivata per progettare il preprocessing iniziale.


## 10. Conclusioni operative

Da questa analisi emergono indicazioni chiare per la fase successiva di preprocessing e baseline modeling.

In primo luogo, `building_id` deve essere escluso dal set di feature predittive, perché rappresenta soltanto un identificativo tecnico dell'edificio.

Le feature geografiche devono essere trattate come variabili categoriche e non come variabili numeriche continue. La loro cardinalità, soprattutto per `geo_level_2_id` e `geo_level_3_id`, richiede attenzione nella scelta dell'encoding.

Le feature numeriche strutturali mostrano un segnale utile ma moderato. Età, numero di piani e altezza tendono ad aumentare con il livello di danno, mentre l'area mostra un andamento inverso. Tuttavia, queste variabili non sembrano sufficienti da sole per separare nettamente le classi.

Le feature categoriche strutturali sono tra le più informative. In particolare, `foundation_type`, `roof_type`, `ground_floor_type` e `other_floor_type` mostrano differenze marcate nella distribuzione di `damage_grade`.

Le feature binarie di superstruttura rappresentano un gruppo centrale per il modello, perché descrivono direttamente materiali e tecniche costruttive. Alcune variabili sono fortemente associate a danni elevati, mentre altre sembrano associate a danni più bassi.

Le feature di uso secondario mostrano un segnale più debole e probabilmente indiretto. Verranno mantenute nella prima pipeline, ma la loro utilità dovrà essere verificata empiricamente.

In sintesi, il preprocessing iniziale dovrà:

- eliminare `building_id`;
- trattare le feature geografiche come categoriche ad alta cardinalità;
- applicare encoding alle feature categoriche strutturali;
- mantenere le feature binarie di superstruttura;
- mantenere inizialmente le feature di uso secondario;
- valutare nella modellazione l'effettivo contributo dei diversi gruppi di feature.

Queste conclusioni guideranno la costruzione della prima pipeline di preprocessing e del modello baseline.
